# CAP Sleep Database — Conversión de archivos .edf a CSV
Convierte todos los archivos `.edf` de `cap-database/` a CSV en `cap-database/edf-converted/`.
Cada archivo generado contiene las señales polisomnográficas: EEG, EOG, EMG, respiración, EKG, etc.

In [1]:
# ── 0. Dependencias ─────────────────────────────────────────────────────────
# Activa el entorno virtual antes de instalar si aún no lo has hecho:
#   source /home/ileana/Documentos/Proyectos/Fusion/myenv/bin/activate
# Luego instala MNE (incluye soporte EDF completo):
#   pip install mne

import sys, os
# Apunta al entorno virtual del proyecto
VENV_SITE = "/home/ileana/Documentos/Proyectos/Fusion/myenv/lib/python3.10/site-packages"
if VENV_SITE not in sys.path:
    sys.path.insert(0, VENV_SITE)

import mne
import pandas as pd
from pathlib import Path
from tqdm.auto import tqdm

print(f"MNE version: {mne.__version__}")

ModuleNotFoundError: No module named 'pandas'

In [ ]:
# ── 1. Rutas ─────────────────────────────────────────────────────────────────
DATA_DIR   = Path("/home/ileana/Documentos/Proyectos/Fusion/data/cap-database")
OUT_DIR    = DATA_DIR / "edf-converted"
OUT_DIR.mkdir(exist_ok=True)

edf_files = sorted(DATA_DIR.glob("*.edf"))
print(f"Archivos .edf encontrados: {len(edf_files)}")
for f in edf_files:
    print(" ", f.name)

In [ ]:
# ── 2. Función de conversión ──────────────────────────────────────────────────
def edf_to_csv(edf_path: Path, out_dir: Path, resample_hz: float = None) -> Path:
    """
    Lee un archivo EDF y lo exporta a CSV.

    Parámetros
    ----------
    edf_path    : ruta al .edf
    out_dir     : carpeta destino
    resample_hz : si se especifica, remuestrea todas las señales a esa
                  frecuencia (útil para reducir el tamaño del CSV).
                  None = mantiene la frecuencia original.

    Retorna
    -------
    Ruta del CSV generado.
    """
    # Carga sin preload para ahorrar RAM; verbose=False silencia los logs de MNE
    raw = mne.io.read_raw_edf(str(edf_path), preload=True, verbose=False)

    if resample_hz is not None:
        raw.resample(resample_hz, verbose=False)

    # Obtiene datos como matriz (n_canales × n_muestras) y los convierte a µV
    data, times = raw.get_data(return_times=True)
    data_uv = data * 1e6  # MNE devuelve en voltios → convertir a µV

    # Construye DataFrame: columna 'time_s' + una columna por canal
    channel_names = raw.ch_names
    df = pd.DataFrame(data_uv.T, columns=channel_names)
    df.insert(0, "time_s", times)

    # Metadatos básicos en las primeras filas del CSV (como comentarios)
    csv_path = out_dir / (edf_path.stem + ".csv")
    meta_lines = [
        f"# subject: {edf_path.stem}",
        f"# sfreq_hz: {raw.info['sfreq']}",
        f"# n_channels: {len(channel_names)}",
        f"# n_samples: {len(times)}",
        f"# duration_s: {times[-1]:.2f}",
        f"# channels: {', '.join(channel_names)}",
    ]
    with open(csv_path, "w") as fh:
        fh.write("\n".join(meta_lines) + "\n")
        df.to_csv(fh, index=False)

    return csv_path

In [ ]:
# ── 3. Conversión en lote ─────────────────────────────────────────────────────
# OPCIONAL: descomenta y ajusta RESAMPLE_HZ para reducir el tamaño de los CSV.
# La frecuencia original de los EDF del CAP suele ser 512 Hz.
# Para análisis de sueño, 256 Hz es suficiente; 128 Hz para señales lentas.
RESAMPLE_HZ = None   # None = sin remuestreo
# RESAMPLE_HZ = 256  # descomenta para remuestrear

errors = []
converted = []

for edf_path in tqdm(edf_files, desc="Convirtiendo EDF → CSV"):
    csv_out = OUT_DIR / (edf_path.stem + ".csv")
    if csv_out.exists():
        print(f"  [ya existe] {csv_out.name} — omitiendo")
        converted.append(csv_out)
        continue
    try:
        csv_path = edf_to_csv(edf_path, OUT_DIR, resample_hz=RESAMPLE_HZ)
        size_mb = csv_path.stat().st_size / 1e6
        print(f"  ✓ {csv_path.name}  ({size_mb:.1f} MB)")
        converted.append(csv_path)
    except Exception as exc:
        print(f"  ✗ {edf_path.name}: {exc}")
        errors.append((edf_path.name, str(exc)))

print(f"\nConvertidos: {len(converted)} / {len(edf_files)}")
if errors:
    print(f"Errores ({len(errors)}):")
    for name, msg in errors:
        print(f"  {name}: {msg}")

In [ ]:
# ── 4. Vista previa del primer CSV generado ───────────────────────────────────
if converted:
    sample_csv = converted[0]
    # Salta las líneas de metadatos (empiezan con '#')
    df_sample = pd.read_csv(sample_csv, comment="#", nrows=5)
    print(f"Vista previa: {sample_csv.name}")
    print(f"Canales: {list(df_sample.columns[1:])}")
    display(df_sample.head())

In [2]:
pip install pandas

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 379.9 kB/s eta 0:00:00 kB/s eta 0:00:0101
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 503.8 kB/s eta 0:00:00m eta 0:00:010:00:01
Note: you may need to restart the kernel to use updated packages.
